In [3]:
import os
import glob
import pandas as pd
import rasterio
import numpy as np

In [ ]:
def preparer_donnees_climat():
   
    base_path = r"Data"
    
   
    df_grille = pd.read_csv("Data\csv_final\grid_portugal_1km.csv")
    print(f"Grille: {len(df_grille)} points")
    
    df_complet = df_grille[['longitude', 'latitude']].copy()
    
    
    dossiers_climat = {
        'Precipitation': 'prec',
        'Tmin': 'tmin', 
        'Tmax': 'tmax'
    }
    
    for dossier, prefixe in dossiers_climat.items():
        chemin_dossier = os.path.join(base_path, "climate", dossier)
        fichiers_tif = glob.glob(os.path.join(chemin_dossier, "*.tif"))
        
        print(f"\n {dossier}: {len(fichiers_tif)} fichiers")
        
        for tif_path in sorted(fichiers_tif):
            nom_fichier = os.path.basename(tif_path)
            mois = nom_fichier.split('_')[-1].replace('.tif', '')
            nom_colonne = f"{prefixe}_{mois}"
            
            print(f"{mois}...", end=" ")
            
            
            valeurs = extraction_vectorisee(tif_path, df_grille)
            df_complet[nom_colonne] = valeurs
            
            valides = np.sum(~np.isnan(valeurs))
            print(f"({valides} valides)")

    output_csv = os.path.join(base_path, "climate\donnees_climat_1km.csv")
    df_complet.to_csv(output_csv, index=False)
    
    return df_complet


def extraction_vectorisee(tif_path, df_points):
    with rasterio.open(tif_path) as src:
        
        rows, cols = rasterio.transform.rowcol(
            src.transform, 
            df_points['longitude'].values, 
            df_points['latitude'].values
        )
        
      
        mask = (rows >= 0) & (rows < src.height) & (cols >= 0) & (cols < src.width)
        
        
        valeurs = np.full(len(df_points), np.nan)
        
        if np.any(mask):
            
            data = src.read(1)
            valeurs_valides = data[rows[mask], cols[mask]]
            
            
            valeurs[mask] = valeurs_valides
            
            
            valeurs[valeurs == src.nodata] = np.nan
    
    return valeurs


donnees_rapides = preparer_donnees_climat()

Grille: 89198 points

 Precipitation: 12 fichiers
2024-01... (89192 valides)
2024-02... (89192 valides)
2024-03... (89192 valides)
2024-04... (89192 valides)
2024-05... (89192 valides)
2024-06... (89192 valides)
2024-07... (89192 valides)
2024-08... (89192 valides)
2024-09... (89192 valides)
2024-10... (89192 valides)
2024-11... (89192 valides)
2024-12... (89192 valides)

 Tmin: 12 fichiers
2024-01... (89192 valides)
2024-02... (89192 valides)
2024-03... (89192 valides)
2024-04... (89192 valides)
2024-05... (89192 valides)
2024-06... (89192 valides)
2024-07... (89192 valides)
2024-08... (89192 valides)
2024-09... (89192 valides)
2024-10... (89192 valides)
2024-11... (89192 valides)
2024-12... (89192 valides)

 Tmax: 12 fichiers
2024-01... (89192 valides)
2024-02... (89192 valides)
2024-03... (89192 valides)
2024-04... (89192 valides)
2024-05... (89192 valides)
2024-06... (89192 valides)
2024-07... (89192 valides)
2024-08... (89192 valides)
2024-09... (89192 valides)
2024-10... (89192 v

In [ ]:
import pandas as pd

df=pd.read_csv(r"Data\climate\donnees_climat_1km.csv")
df.head()

def transformer_mois_en_saisons(df_climat):
  
    print("TRANSFORMATION MOIS en SAISONS")
    
    
   
    df_saisons = df_climat[['longitude', 'latitude']].copy()
    
   
    saisons = {
        'hiver': ['01', '02', '12'],    
        'printemps': ['03', '04', '05'], 
        'ete': ['06', '07', '08'],        
        'automne': ['09', '10', '11']   
    }
    
    variables = ['prec', 'tmin', 'tmax']
    
    for variable in variables:
        print(f"\n raitement de: {variable}")
        
        
        colonnes_variable = [col for col in df_climat.columns if col.startswith(f"{variable}_")]
        print(f"Colonnes mensuelles: {len(colonnes_variable)}")
        
        for saison, mois_list in saisons.items():
            
            colonnes_saison = [col for col in colonnes_variable if any(f"-{m}" in col for m in mois_list)]
            
            if len(colonnes_saison) == 3:  
                print(f" {saison}: {colonnes_saison}")
                
                
                df_saisons[f'{variable}_{saison}'] = df_climat[colonnes_saison].mean(axis=1)
            else:
                print(f" {saison}: {len(colonnes_saison)} mois trouvés (attendu: 3)")
    
  
    colonnes_originales = len([c for c in df_climat.columns if any(v in c for v in ['prec', 'tmin', 'tmax'])])
    colonnes_saisonnières = len([c for c in df_saisons.columns if any(v in c for v in ['prec', 'tmin', 'tmax'])])
    
    print(f"\nTRANSFORMATION TERMINÉE:")
    print(f"Colonnes originales (mensuelles): {colonnes_originales}")
    print(f"Colonnes saisonnières: {colonnes_saisonnières}")
    print(f"Réduction: {colonnes_originales} → {colonnes_saisonnières} colonnes")
    
    return df_saisons


df_saisons = transformer_mois_en_saisons(df)
df_saisons.head()

df_saisons.to_csv(r"Data\climate\climat.csv", index=False)

TRANSFORMATION MOIS en SAISONS

 raitement de: prec
Colonnes mensuelles: 12
 hiver: ['prec_2024-01', 'prec_2024-02', 'prec_2024-12']
 printemps: ['prec_2024-03', 'prec_2024-04', 'prec_2024-05']
 ete: ['prec_2024-06', 'prec_2024-07', 'prec_2024-08']
 automne: ['prec_2024-09', 'prec_2024-10', 'prec_2024-11']

 raitement de: tmin
Colonnes mensuelles: 12
 hiver: ['tmin_2024-01', 'tmin_2024-02', 'tmin_2024-12']
 printemps: ['tmin_2024-03', 'tmin_2024-04', 'tmin_2024-05']
 ete: ['tmin_2024-06', 'tmin_2024-07', 'tmin_2024-08']
 automne: ['tmin_2024-09', 'tmin_2024-10', 'tmin_2024-11']

 raitement de: tmax
Colonnes mensuelles: 12
 hiver: ['tmax_2024-01', 'tmax_2024-02', 'tmax_2024-12']
 printemps: ['tmax_2024-03', 'tmax_2024-04', 'tmax_2024-05']
 ete: ['tmax_2024-06', 'tmax_2024-07', 'tmax_2024-08']
 automne: ['tmax_2024-09', 'tmax_2024-10', 'tmax_2024-11']

TRANSFORMATION TERMINÉE:
Colonnes originales (mensuelles): 36
Colonnes saisonnières: 12
Réduction: 36 → 12 colonnes


In [ ]:
df_climat = pd.read_csv(r"Data\climate\climat.csv")

print("Arrondissement des coordonnées à 3 décimales...")

df_climat['latitude'] = df_climat['latitude'].round(3)
df_climat['longitude'] = df_climat['longitude'].round(3)

print("Recherche des lignes duplicates...")
masque_doublons = df_climat.duplicated(subset=['latitude', 'longitude'], keep=False)

df_doublons = df_climat[masque_doublons].copy()
df_climat.to_csv(r"Data\csv_final\climat.csv", index=False)

        
if df_doublons.empty:
            print("Bonne nouvelle : Aucune ligne duplicate trouvée après arrondissement.")
else:
            print(f"Trouvé {len(df_doublons)} lignes qui sont des doublons.")
            
           
            df_doublons.sort_values(by=['latitude', 'longitude'], inplace=True)

Arrondissement des coordonnées à 3 décimales...
Recherche des lignes duplicates...
Bonne nouvelle : Aucune ligne duplicate trouvée après arrondissement.
